<a href="https://colab.research.google.com/github/DavidDevKing/extractive-summarization-system-for-medical-reports/blob/main/notebooks/inference_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
import torch

# This creates a folder called 'drive' in your Colab sidebar
drive.mount('/content/drive')

# Define exactly where your model is stored
# Make sure this matches where you saved it in the training notebook!
MODEL_PATH = "/content/drive/MyDrive/School/CSC 309 Colab Notebooks/med_summarizer_trained.pt"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!git clone "https://github.com/DavidDevKing/extractive-summarization-system-for-medical-reports.git"
!pip install -r extractive-summarization-system-for-medical-reports/notebooks/requirements.txt

import sys
sys.path.append("extractive-summarization-system-for-medical-reports/notebooks/")
from utils import *

import os
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import spacy

# --- PART 1: THE ARCHITECTURE ---
class BioExtractor(nn.Module):
    def __init__(self, model_name):
        super(BioExtractor, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(768, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.sigmoid(self.classifier(cls_output))

# --- PART 2: THE LOADING LOGIC ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
nlp = spacy.load("en_core_web_sm")

# Initialize the "Body"
model = BioExtractor(model_name)

# Load the "Brain" from Google Drive
if os.path.exists(MODEL_PATH):
    print("Loading trained weights from Google Drive...")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print("Model Loaded Successfully!")
else:
    print(f"ERROR: Could not find the model at {MODEL_PATH}")
    print("Check if you saved it to the correct folder in the Training Notebook.")

model.to(device)
model.eval()

# --- PART 3: THE INFERENCE FUNCTION ---
def run_prediction(text, top_k=3):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent.text) > 10]

    scores = []
    with torch.no_grad():
        for sent in sentences:
            inputs = tokenizer(sent, return_tensors="pt", truncation=True, padding='max_length', max_length=128).to(device)
            output = model(inputs['input_ids'], inputs['attention_mask'])
            scores.append(output.item())

    # Sort and pick top sentences
    scored_sentences = sorted(zip(scores, sentences), key=lambda x: x[0], reverse=True)
    return " ".join([s for _, s in scored_sentences[:top_k]])






# Test it
def main():
    # Generate the summary
    sample_text = "Patient displays symptoms of chronic bronchitis. Sputum culture was negative. Recommend immediate follow-up with a pulmonologist."
    print("\n--- SUMMARY ---")
    summary = run_prediction(sample_text, top_k=3)
    print(summary)


    # Generate the scores
    print("---SCORES---")
    reference_summary = "Patient displays symptoms of chronic bronchitis. Sputum culture was negative. Recommend immediate follow-up with a pulmonologist."
    print(extract_scores(reference_summary, summary))
    print ("_"*50)

if __name__ == "__main__":
    main()








fatal: destination path 'extractive-summarization-system-for-medical-reports' already exists and is not an empty directory.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.2 MB/s eta 0:00:00
Loading trained weights from Google Drive...
Model Loaded Successfully!

--- SUMMARY ---
Sputum culture was negative. Recommend immediate follow-up with a pulmonologist. Patient displays symptoms of chronic bronchitis.
---SCORES---
rouge1: F1-Score = 1.0000
rouge1: Precision = 1.0000
rouge1: Recall = 1.0000

rouge2: F1-Score = 0.9375
rouge2: Precision = 0.9375
rouge2: Recall = 0.9375

rougeL: F1-Score = 0.6471
rougeL: Precision = 0.6471
rougeL: Recall = 0.6471
__________________________________________________


In [5]:
from datasets import load_dataset
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

# 1. Load the Test Set (Records the model has NEVER seen)
test_dataset = load_dataset("ccdv/pubmed-summarization", "section", split="test", streaming=True)
test_records = list(test_dataset.take(50)) # Testing on 50 records is statistically solid

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
results = {'rouge1': [], 'rouge2' : [] 'rougeL': []}

print("Evaluating Trained Model on 50 Test Records...")

for entry in tqdm(test_records):
    article = entry['article']
    reference = entry['abstract']

    # Generate summary using your trained model function
    # We use top_k=3 to match the standard abstract length
    generated_summary = run_prediction(article, top_k=3)

    # Calculate Scores
    score = scorer.score(reference, generated_summary)
    results['rouge1'].append(score['rouge1'].fmeasure)
    results['rouge2'].append(score['rouge2'].fmeasure)
    results['rougeL'].append(score['rougeL'].fmeasure)

# 2. CALCULATE AVERAGES
avg_r1 = sum(results['rouge1']) / len(results['rouge1'])
avg_r2 = sum(results['rouge2']) / len(results['rouge2'])
avg_rL = sum(results['rougeL']) / len(results['rougeL'])

print("\n--- TRAINED MODEL FINAL SCORES ---")
print(f"Average ROUGE-1: {avg_r1:.4f}")
print(f"Average ROUGE-2: {avg_r2:.4f}")
print(f"Average ROUGE-L: {avg_rL:.4f}")

README.md: 0.00B [00:00, ?B/s]

Evaluating Trained Model on 50 Test Records...


  0%|          | 0/50 [00:00<?, ?it/s]


--- TRAINED MODEL FINAL SCORES ---
rouge1: F1-Score = 0.3782
rouge1: Precision = 0.4370
rouge1: Recall = 0.3333

rouge2: F1-Score = 0.0586
rouge2: Precision = 0.0678
rouge2: Recall = 0.0516

rougeL: F1-Score = 0.1600
rougeL: Precision = 0.1849
rougeL: Recall = 0.1410
